# Backward pass of a regression network

A network predicts three numbers per house with leaky ReLU hidden layers and a linear output,
and it is trained on the mean squared error as PyTorch computes it by default. You write the
error at the output and the backward pass for any depth in NumPy, and check every gradient
against central differences.

**How to work through it.** Each part asks for one function. Replace the `TODO` and the
`raise NotImplementedError` with your code, then run the check cell under it. The check
prints `Part N passed` when your function is right, and stops with a message that says
what is wrong when it is not.

## Setup

Run this cell first.

In [ ]:
import numpy as np

def leaky_relu(Z):
    return np.where(Z > 0, Z, 0.1 * Z)

def forward(X, params):
    """Leaky ReLU hidden layers and a linear output layer. Rows are examples."""
    cache = {"A0": X}
    A = X
    L = len(params)
    for l, (W, b) in enumerate(params, start=1):
        Z = A @ W + b
        A = Z if l == L else leaky_relu(Z)
        cache[f"Z{l}"], cache[f"A{l}"] = Z, A
    return A, cache

def mse(Yhat, Y):
    """The mean of the squared errors over every entry, the default of PyTorch's nn.MSELoss."""
    return np.mean((Yhat - Y) ** 2)

def numeric_grad(f, A, h=1e-6):
    """Central differences of f() in every entry of the array A. A is nudged in place and restored."""
    G = np.zeros_like(A)
    for idx in np.ndindex(A.shape):
        old = A[idx]
        A[idx] = old + h
        up = f()
        A[idx] = old - h
        down = f()
        A[idx] = old
        G[idx] = (up - down) / (2 * h)
    return G

# Six houses, three features each, three targets per house (price, rent and tax, rescaled).
seed = np.random.default_rng(7)
X = seed.normal(size=(6, 3))
Y = seed.normal(size=(6, 3))
params = [
    (seed.normal(size=(3, 4)), 0.5 * seed.normal(size=4)),
    (0.8 * seed.normal(size=(4, 4)), 0.5 * seed.normal(size=4)),
    (seed.normal(size=(4, 3)), seed.normal(size=3)),
]
Yhat, cache = forward(X, params)
print("loss", round(mse(Yhat, Y), 4))

## Part 1 (coding task): The error at the output

The output layer is linear, $\hat Y = A^{(2)} W^{(3)} + \mathbf{b}^{(3)}$, so its pre-activation
is $\hat Y$ itself. The loss is `mse`, the mean of $(\hat Y - Y)^2$ over all $N \times K$ entries of
the $N \times K$ target array, with no factor $\tfrac12$.

Write `output_error(Yhat, Y)`, which returns $\partial L / \partial \hat Y$, an array shaped like `Yhat`.

In [ ]:
def output_error(Yhat, Y):
    """The gradient of mse(Yhat, Y) with respect to Yhat."""
    # TODO
    raise NotImplementedError

In [ ]:
G = output_error(Yhat, Y)
assert G.shape == Yhat.shape, f"expected shape {Yhat.shape}, got {G.shape}"
Yh = Yhat.copy()
num = numeric_grad(lambda: mse(Yh, Y), Yh)
ratio = np.sum(G * num) / np.sum(num * num)
if not np.allclose(G, num, atol=1e-7):
    if np.allclose(G, ratio * num, atol=1e-7):
        raise AssertionError(f"right direction, but {ratio:.3g} times the true gradient: "
                             "check the factor from the square and what the mean divides by")
    raise AssertionError("output_error does not match the numeric gradient of mse")
print("Part 1 passed")

## Part 2 (coding task): The backward pass

Write `backward(cache, params, Y)`. It returns a list with one pair `(dW, db)` per layer, in
layer order, holding the gradients of `mse` with respect to that layer's weights and bias, each
shaped like its parameter. Use `output_error` and the cache that `forward` returns, and work
for any number of hidden layers. The slope of the leaky ReLU is 1 where $z > 0$ and 0.1 where
$z \le 0$.

In [ ]:
def backward(cache, params, Y):
    """[(dW, db)] for each layer, gradients of mse with respect to that layer's parameters."""
    # TODO
    raise NotImplementedError

In [ ]:
grads = backward(cache, params, Y)
assert len(grads) == len(params), f"expected {len(params)} (dW, db) pairs, got {len(grads)}"

def loss_now():
    return mse(forward(X, params)[0], Y)

for l in range(len(params), 0, -1):
    (W, b), (dW, db) = params[l - 1], grads[l - 1]
    assert np.shape(dW) == W.shape, f"layer {l}: dW has shape {np.shape(dW)}, W has {W.shape}"
    assert np.shape(db) == b.shape, f"layer {l}: db has shape {np.shape(db)}, b has {b.shape}"
    for name, P, G in (("dW", W, dW), ("db", b, db)):
        num = numeric_grad(loss_now, P)
        if not np.allclose(G, num, atol=1e-6):
            hint = ("the output error or the outer products" if l == len(params)
                    else "the step back into this layer: the transpose, and the leaky ReLU slope on both sides of 0")
            raise AssertionError(f"layer {l} {name} does not match the numeric gradient; look at {hint}")

# A deeper network: four hidden layers of width 5.
rng2 = np.random.default_rng(11)
deep = [(rng2.normal(size=(3, 5)), rng2.normal(size=5))] + \
       [(0.7 * rng2.normal(size=(5, 5)), rng2.normal(size=5)) for _ in range(3)] + \
       [(rng2.normal(size=(5, 3)), rng2.normal(size=3))]
_, deep_cache = forward(X, deep)
deep_grads = backward(deep_cache, deep, Y)
dW1_num = numeric_grad(lambda: mse(forward(X, deep)[0], Y), deep[0][0])
assert np.allclose(deep_grads[0][0], dW1_num, atol=1e-6), "right on three layers but wrong on five: is the loop tied to one depth?"
print("Part 2 passed")

## Solution

### Part 1: The error at the output

In [ ]:
def output_error(Yhat, Y):
    """The gradient of mse(Yhat, Y) with respect to Yhat."""
    N, K = Y.shape
    return 2 * (Yhat - Y) / (N * K)

**Why it works.** Each entry $(\hat y_{nk} - y_{nk})^2$ has derivative $2(\hat y_{nk} - y_{nk})$, and the mean
divides the sum of all $NK$ entries by $NK$. The output unit is linear, so this is already the error
at the pre-activation. Two slips give the right direction at the wrong size: dropping the 2, which
belongs to losses written with a $\tfrac12$, and dividing by $N$ alone, which is the gradient of a
loss summed over the targets and averaged over the examples. Both pass every sign test and only
change the effective learning rate, which is why the check compares against the numbers.

### Part 2: The backward pass

In [ ]:
def backward(cache, params, Y):
    """[(dW, db)] for each layer, gradients of mse with respect to that layer's parameters."""
    L = len(params)
    grads = [None] * L
    delta = output_error(cache[f"A{L}"], Y)
    for l in range(L, 0, -1):
        W, b = params[l - 1]
        grads[l - 1] = (cache[f"A{l-1}"].T @ delta, delta.sum(axis=0))
        if l > 1:
            Z = cache[f"Z{l-1}"]
            delta = (delta @ W.T) * np.where(Z > 0, 1.0, 0.1)
    return grads

**Why it works.** The loop is the lesson's backward pass with two changes. The error starts from
`output_error` instead of $\hat Y - Y$ divided by $N$, and the step back multiplies by the leaky
ReLU's slope, which is 0.1 rather than 0 for units below zero, so a unit that is off still passes a
tenth of its error back. Each layer's gradients are `A_prev.T @ delta`, the outer products summed
over the rows, and `delta.sum(axis=0)`. The slope is evaluated at $Z^{(l-1)}$, the pre-activation
of the layer the error is entering, not the one it is leaving.